# 🔮 Previsões Futuras e Validação do Modelo

Este notebook é dedicado a demonstrar na prática como o modelo de regressão treinado pode ser utilizado para fazer previsões e simulações de cenários futuros.

Aqui nós:
1. Carregamos a base unificada.
2. Filtramos as variáveis e aplicamos a codificação binária (Binary Encoding) idêntica à do modelo final.
3. Ajustamos o modelo OLS (Mínimos Quadrados Ordinários).
4. Geramos previsões de instabilidade financeira (`target_instabilidade`) para as secretarias municipais.
5. Comparamos o valor real com o previsto para mensurar a acurácia (precisão) do modelo.

In [ ]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Configuração de caminhos
base_dir = 'bases' if os.path.exists('bases') else '../bases'
df_path = os.path.join(base_dir, 'base_unificada.csv')
df = pd.read_csv(df_path, low_memory=False)
print(f"Base unificada carregada com sucesso. Total de linhas: {df.shape[0]}")

## 🛠️ Preparação dos Dados e Codificação em Bits

In [ ]:
# Seleção das variáveis preditoras (X) e alvo (Y)
variaveis_x = [
    'de_tipoEmpenho',
    'de_idRecurso',
    'de_categoriaEmpenho',
    'de_idElemento',
    'de_modalidadeAplicacao',
    'de_tipoRecurso',
    'de_idProjetoAtividade',
    'de_idPrograma',
    'de_idFuncao',
    'de_descricaoUnidade'
]
variavel_y = 'target_instabilidade'

# Filtramos nulos mantendo o mapeamento de órgãos e datas para visualização
df_modelo = df[variaveis_x + [variavel_y, 'orgao_padronizado', 'mes_ano']].dropna()
print(f"Registros válidos para previsão: {df_modelo.shape[0]}")

X = df_modelo[variaveis_x].copy()
Y = df_modelo[variavel_y].astype(float)

# Aplicação de Binary Encoding (Bits)
colunas_originais = list(X.columns)
for col in colunas_originais:
    X[col] = X[col].astype('category')
    codigos = X[col].cat.codes
    num_categorias = len(X[col].cat.categories)
    
    if num_categorias > 1:
        num_bits = int(np.ceil(np.log2(num_categorias)))
        for i in range(num_bits):
            X[f'{col}_Bit{i+1}'] = (codigos // (2**i)) % 2
    X = X.drop(col, axis=1)

X = X.astype(float)
X = sm.add_constant(X)
print(f"Dimensões da matriz preditora X: {X.shape}")

## 🧠 Ajuste do Modelo de Regressão Múltipla

In [ ]:
modelo = sm.OLS(Y, X)
resultado = modelo.fit()
print("Modelo OLS ajustado com sucesso!")

## 🔮 Geração de Previsões e Validação de Acurácia

Calculamos as previsões e medimos o erro percentual absoluto médio de cada previsão para validar a proximidade com a realidade.

In [ ]:
# Realizando as previsões na base
df_modelo['previsao'] = resultado.predict(X)
df_modelo['erro_absoluto'] = (df_modelo['previsao'] - df_modelo['target_instabilidade']).abs()

# Evitando divisão por zero no cálculo do percentual
df_modelo['erro_abs_pct'] = (df_modelo['erro_absoluto'] / df_modelo['target_instabilidade'].replace(0, 1)) * 100
df_modelo['acuracia_pct'] = 100 - df_modelo['erro_abs_pct']

# Limitando a acurácia mínima em 0% (caso a previsão erre por mais de 100% o valor real)
df_modelo['acuracia_pct'] = df_modelo['acuracia_pct'].clip(lower=0)

print("Métricas de erro calculadas!")

## 📊 Exibição de Exemplos Reais para a Apresentação

Abaixo selecionamos alguns exemplos significativos de secretarias chave em períodos reais da série histórica para demonstrar a proximidade entre a estimativa e o real.

In [ ]:
# Agrupamos os exemplos por órgão
exemplos = df_modelo.groupby('orgao_padronizado').first().reset_index()

print("=================== EXEMPLOS DE PREVISÃO DO MODELO ===================\n")
for index, row in exemplos.head(5).iterrows():
    print(f"🏢 Órgão/Secretaria: {row['orgao_padronizado']}")
    print(f"📅 Mês/Ano: {row['mes_ano']}")
    print(f"📝 Tipo de Empenho: {row['de_tipoEmpenho']}")
    print(f"💰 Instabilidade Real: R$ {row['target_instabilidade']:,.2f}")
    print(f"🔮 Instabilidade Prevista: R$ {row['previsao']:,.2f}")
    print(f"🎯 Acurácia do Modelo: {row['acuracia_pct']:.2f}%")
    print("-" * 70)